In [ ]:
import cv2
import numpy as np
import glob
import os
import pandas as pd
from skimage.feature import local_binary_pattern
from scipy.stats import skew

# --- Parameters ---
GRID_ROWS, GRID_COLS = 8, 8
TARGET_W, TARGET_H = 800, 600
SUBGRID_SIZE = 4  # subdivide each grid block
OUTFILE = "compact_block_features.csv"

# --- Preprocessing ---
def enforce_4_3_aspect_and_scale(img):
    h, w = img.shape[:2]
    desired_ratio = 4 / 3
    cur_ratio = w / h
    if abs(cur_ratio - desired_ratio) > 1e-3:
        if cur_ratio > desired_ratio:
            new_w = int(h * desired_ratio)
            x0 = (w - new_w) // 2
            img = img[:, x0:x0 + new_w]
        else:
            new_h = int(w / desired_ratio)
            y0 = (h - new_h) // 2
            img = img[y0:y0 + new_h, :]
    img = cv2.resize(img, (TARGET_W, TARGET_H))
    return img

# --- Grid Splitter ---
def split_to_grids(img, rows=GRID_ROWS, cols=GRID_COLS):
    h, w = img.shape[:2]
    ch, cw = h // rows, w // cols
    grids = []
    for r in range(rows):
        for c in range(cols):
            y0, x0 = r * ch, c * cw
            grids.append(img[y0:y0 + ch, x0:x0 + cw])
    return grids

# --- Feature Extraction ---
def extract_block_features(block):
    gray = cv2.cvtColor(block, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(block, cv2.COLOR_BGR2HSV)
    feats = []

    # --- Sobel features (edge info) ---
    sx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    sy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    smag = np.sqrt(sx ** 2 + sy ** 2)
    feats += [sx.mean(), sy.mean(), smag.mean(), sx.std(), sy.std(), smag.std()]

    # --- Laplacian (texture variation) ---
    lap = cv2.Laplacian(gray, cv2.CV_32F)
    feats += [lap.mean(), lap.std(), skew(lap.ravel())]

    # --- LBP (texture pattern) ---
    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, 10), range=(0, 9))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)
    feats += hist.tolist()

    # --- HSV mean + std ---
    feats += [hsv[..., 0].mean(), hsv[..., 1].mean(), hsv[..., 2].mean(),
              hsv[..., 0].std(), hsv[..., 1].std(), hsv[..., 2].std()]

    return np.array(feats, dtype=np.float32)

# --- Main Loop ---
all_feats = []
img_dir = "images"
image_paths = []
for ext in ("*.jpg", "*.jpeg", "*.JPG", "*.png"):
    image_paths.extend(glob.glob(os.path.join(img_dir, ext)))
image_paths = sorted(image_paths)

for img_path in image_paths:
    img = cv2.imread(img_path)
    if img is None:
        continue
    img = enforce_4_3_aspect_and_scale(img)
    grids = split_to_grids(img)
    for idx, g in enumerate(grids):
        feats = extract_block_features(g)
        all_feats.append([os.path.basename(img_path), idx] + feats.tolist())

# --- Save to CSV ---
columns = ["filename", "grid_index"] + [f"feat_{i}" for i in range(len(all_feats[0]) - 2)]
df = pd.DataFrame(all_feats, columns=columns)
df.to_csv(OUTFILE, index=False)
print(f"✅ Saved compact features for {len(image_paths)} images and {len(df)} blocks → {OUTFILE}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

# --- LOAD FEATURES ---
features_df = pd.read_csv("compact_block_features.csv")

# --- LOAD LABELS ---
labels_df = pd.read_excel("labels_final.xlsx")  # Excel: filename + block_0..block_63

# --- Convert labels to long format ---
labels_long = labels_df.melt(
    id_vars="filename",
    var_name="block",
    value_name="label"
)
labels_long["grid_index"] = labels_long["block"].str.extract(r"(\d+)").astype(int)

# --- Merge features with labels ---
merged = features_df.merge(labels_long, on=["filename", "grid_index"])
print(f"✅ Merged dataset shape: {merged.shape}")

# --- Prepare X and y ---
feature_cols = [c for c in merged.columns if c.startswith("feat_")]
X = merged[feature_cols]
y = merged["label"]

# --- Ensure no NaNs ---
X = X.fillna(X.mean())

# --- Normalize features ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# --- Handle class imbalance for binary classification ---
if len(np.unique(y)) == 2:
    neg, pos = np.bincount(y_train)
    scale_pos_weight = neg / pos
else:
    scale_pos_weight = 1.0  # multi-class, ignored by XGBoost

# --- Train XGBoost ---
model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)
model.fit(X_train, y_train)

# --- Predict ---
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
y_train_proba = model.predict_proba(X_train)
y_test_proba = model.predict_proba(X_test)

# --- Train/Test Metrics ---
print("\n📊 Train Classification Report:")
print(classification_report(y_train, y_train_pred))
print("\n📊 Test Classification Report:")
print(classification_report(y_test, y_test_pred))

# --- Confusion Matrices ---
fig, axs = plt.subplots(1, 2, figsize=(12,5))
sns.heatmap(confusion_matrix(y_train, y_train_pred), annot=True, fmt='d', cmap='Blues', ax=axs[0])
axs[0].set_title("Train Confusion Matrix")
axs[0].set_xlabel("Predicted")
axs[0].set_ylabel("Actual")
sns.heatmap(confusion_matrix(y_test, y_test_pred), annot=True, fmt='d', cmap='Blues', ax=axs[1])
axs[1].set_title("Test Confusion Matrix")
axs[1].set_xlabel("Predicted")
axs[1].set_ylabel("Actual")
plt.show()

# --- ROC AUC Curves ---
if len(np.unique(y)) == 2:
    # Binary
    fpr_train, tpr_train, _ = roc_curve(y_train, y_train_proba[:,1])
    fpr_test, tpr_test, _ = roc_curve(y_test, y_test_proba[:,1])
    auc_train = auc(fpr_train, tpr_train)
    auc_test = auc(fpr_test, tpr_test)
    
    plt.figure(figsize=(8,6))
    plt.plot(fpr_train, tpr_train, label=f"Train ROC (AUC={auc_train:.3f})")
    plt.plot(fpr_test, tpr_test, label=f"Test ROC (AUC={auc_test:.3f})")
    plt.plot([0,1], [0,1], 'k--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.show()
else:
    # Multi-class (One-vs-Rest)
    y_train_bin = label_binarize(y_train, classes=np.unique(y))
    y_test_bin = label_binarize(y_test, classes=np.unique(y))
    n_classes = y_train_bin.shape[1]
    
    plt.figure(figsize=(8,6))
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_test_proba[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"Class {i} (AUC={roc_auc:.2f})")
    plt.plot([0,1], [0,1], 'k--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Multi-class ROC")
    plt.legend()
    plt.show()

# --- Save metrics to Excel ---
report_train = classification_report(y_train, y_train_pred, output_dict=True)
report_test = classification_report(y_test, y_test_pred, output_dict=True)
metrics_df = pd.concat([
    pd.DataFrame(report_train).transpose().assign(dataset='train'),
    pd.DataFrame(report_test).transpose().assign(dataset='test')
])
metrics_df.to_excel("XGBoost_Block_Metrics_TrainTest.xlsx")
print("📁 Saved train/test metrics to 'XGBoost_Block_Metrics_TrainTest.xlsx'")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# --- PARAMETERS ---
GRID_ROWS, GRID_COLS = 8, 8
TARGET_W, TARGET_H = 800, 600
save_dir = "xgboost_predictions_highlighted"
os.makedirs(save_dir, exist_ok=True)

def enforce_4_3_aspect_and_scale(img):
    h, w = img.shape[:2]
    desired_ratio = 4 / 3
    cur_ratio = w / h
    if abs(cur_ratio - desired_ratio) > 1e-3:
        if cur_ratio > desired_ratio:
            new_w = int(h * desired_ratio)
            x0 = (w - new_w) // 2
            img = img[:, x0:x0 + new_w]
        else:
            new_h = int(w / desired_ratio)
            y0 = (h - new_h) // 2
            img = img[y0:y0 + new_h, :]
    img = cv2.resize(img, (TARGET_W, TARGET_H))
    return img

def highlight_blocks(img, labels, alpha=0.4):
    h, w = img.shape[:2]
    cell_h, cell_w = h // GRID_ROWS, w // GRID_COLS
    img_out = img.copy()
    for i in range(GRID_ROWS):
        for j in range(GRID_COLS):
            idx = i * GRID_COLS + j
            if labels[idx] == 1:
                y0, y1 = i * cell_h, (i + 1) * cell_h
                x0, x1 = j * cell_w, (j + 1) * cell_w
                overlay = img_out.copy()
                overlay[y0:y1, x0:x1] = (0, 0, 255)  # red overlay
                cv2.addWeighted(overlay, alpha, img_out, 1 - alpha, 0, img_out)
    return img_out

# --- Get unique image names ---
image_names = merged["filename"].unique()

for fname in image_names:
    img_path = os.path.join("images", fname)
    img = cv2.imread(img_path)
    if img is None:
        continue
    img = enforce_4_3_aspect_and_scale(img)

    # Extract feature rows for this image
    sub_df = merged[merged["filename"] == fname]
    X_blocks = scaler.transform(sub_df[[c for c in merged.columns if c.startswith("feat_")]])
    
    # Predict each block
    preds = model.predict(X_blocks)
    
    # Ensure we have 64 predictions
    if len(preds) < 64:
        preds = np.pad(preds, (0, 64 - len(preds)), constant_values=0)
    elif len(preds) > 64:
        preds = preds[:64]
    
    # Highlight
    highlighted = highlight_blocks(img, preds)
    
    # Display
    plt.figure(figsize=(8,6))
    plt.imshow(cv2.cvtColor(highlighted, cv2.COLOR_BGR2RGB))
    plt.title(f"Predicted Animal Blocks: {fname}")
    plt.axis('off')
    plt.show()
    
    # Save
    save_path = os.path.join(save_dir, fname)
    cv2.imwrite(save_path, highlighted)
    print(f"✅ Saved: {save_path}")


In [ ]:
import cv2
import numpy as np
import glob
import os
import pandas as pd
from skimage.feature import local_binary_pattern
from scipy.stats import skew

# --- Parameters ---
GRID_ROWS, GRID_COLS = 8, 8
TARGET_W, TARGET_H = 800, 600
OUT_DIR = "features"
os.makedirs(OUT_DIR, exist_ok=True)

# Compact Gabor setup: 4 orientations × 2 scales
GABOR_THETAS = [0, np.pi/4, np.pi/2, 3*np.pi/4]
GABOR_SIGMAS = [1, 2]

In [ ]:
# --- Helper: Resize and preserve 4:3 ratio ---
def enforce_4_3_aspect_and_scale(img):
    h, w = img.shape[:2]
    desired_ratio = 4 / 3
    cur_ratio = w / h
    if abs(cur_ratio - desired_ratio) > 1e-3:
        if cur_ratio > desired_ratio:
            new_w = int(h * desired_ratio)
            x0 = (w - new_w) // 2
            img = img[:, x0:x0 + new_w]
        else:
            new_h = int(w / desired_ratio)
            y0 = (h - new_h) // 2
            img = img[y0:y0 + new_h, :]
    return cv2.resize(img, (TARGET_W, TARGET_H))

# --- Split into 8×8 grids ---
def split_to_grids(img, rows=GRID_ROWS, cols=GRID_COLS):
    h, w = img.shape[:2]
    ch, cw = h // rows, w // cols
    return [img[r*ch:(r+1)*ch, c*cw:(c+1)*cw] for r in range(rows) for c in range(cols)]

# --- Core extractor (choose feature subset) ---
def extract_block_features(block, use_sobel=True, use_laplacian=True, use_lbp=True, use_hsv=True, use_gabor=True):
    gray = cv2.cvtColor(block, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(block, cv2.COLOR_BGR2HSV)
    feats = []

    # --- Sobel ---
    if use_sobel:
        sx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
        sy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
        smag = np.sqrt(sx**2 + sy**2)
        feats += [sx.mean(), sy.mean(), smag.mean(), sx.std(), sy.std(), smag.std()]

    # --- Laplacian ---
    if use_laplacian:
        lap = cv2.Laplacian(gray, cv2.CV_32F)
        feats += [lap.mean(), lap.std(), skew(lap.ravel())]

    # --- LBP ---
    if use_lbp:
        lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
        (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, 10), range=(0, 9))
        hist = hist.astype("float")
        hist /= (hist.sum() + 1e-6)
        feats += hist.tolist()

    # --- HSV mean/std ---
    if use_hsv:
        feats += [hsv[...,0].mean(), hsv[...,1].mean(), hsv[...,2].mean(),
                  hsv[...,0].std(), hsv[...,1].std(), hsv[...,2].std()]

    # --- Gabor (compact: 4 orientations × 2 scales) ---
    if use_gabor:
        for theta in GABOR_THETAS:
            for sigma in GABOR_SIGMAS:
                kernel = cv2.getGaborKernel((9, 9), sigma, theta, 8, 0.5, 0, ktype=cv2.CV_32F)
                filtered = cv2.filter2D(gray, cv2.CV_32F, kernel)
                feats += [filtered.mean(), filtered.std()]

    return np.array(feats, dtype=np.float32)


In [ ]:
def extract_and_save_features(feature_name, img_dir="images", **kwargs):
    all_feats = []
    for ext in ("*.jpg", "*.jpeg", "*.JPG", "*.png"):
        for img_path in glob.glob(os.path.join(img_dir, ext)):
            img = cv2.imread(img_path)
            if img is None:
                continue
            img = enforce_4_3_aspect_and_scale(img)
            grids = split_to_grids(img)
            for idx, g in enumerate(grids):
                feats = extract_block_features(g, **kwargs)
                all_feats.append([os.path.basename(img_path), idx] + feats.tolist())

    cols = ["filename", "grid_index"] + [f"feat_{i}" for i in range(len(all_feats[0]) - 2)]
    df = pd.DataFrame(all_feats, columns=cols)
    out_path = os.path.join(OUT_DIR, f"{feature_name}_features.csv")
    df.to_csv(out_path, index=False)
    print(f"✅ Saved {feature_name} features → {out_path}")

# ---- Run all ----
extract_and_save_features("all", use_sobel=True, use_laplacian=True, use_lbp=True, use_hsv=True, use_gabor=True)
extract_and_save_features("sobel", use_sobel=True, use_laplacian=False, use_lbp=False, use_hsv=False, use_gabor=False)
extract_and_save_features("laplacian", use_sobel=False, use_laplacian=True, use_lbp=False, use_hsv=False, use_gabor=False)
extract_and_save_features("lbp", use_sobel=False, use_laplacian=False, use_lbp=True, use_hsv=False, use_gabor=False)
extract_and_save_features("hsv", use_sobel=False, use_laplacian=False, use_lbp=False, use_hsv=True, use_gabor=False)
extract_and_save_features("gabor", use_sobel=False, use_laplacian=False, use_lbp=False, use_hsv=False, use_gabor=True)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from xgboost import XGBClassifier
import glob, os

# --- Paths ---
FEATURE_DIR = "features"                   # Folder with CSV feature files
LABEL_FILE = "labels_final.xlsx"           # Excel file with 64-block labels
label_df = pd.read_excel(LABEL_FILE)

# Ensure filename is string
label_df["filename"] = label_df["filename"].astype(str)

# --- Helper function ---
def evaluate_feature_set(csv_path, label_df):
    feature_name = os.path.basename(csv_path).replace("_features.csv", "")
    print(f"\n🔹 Evaluating feature set: {feature_name.upper()}")

    # Load features
    df = pd.read_csv(csv_path)
    df["filename"] = df["filename"].astype(str)

    # Merge labels
    merged = df.merge(label_df, on="filename", how="inner")
    if merged.empty:
        print(f"❌ No matching filenames in labels for {csv_path}, skipping.")
        return None

    # Identify label columns
    y_cols = [c for c in merged.columns if c.startswith("block_")]

    # Identify feature columns (all numeric except filename and label columns)
    X_cols = [c for c in merged.select_dtypes(include=np.number).columns if c not in y_cols]
    if len(X_cols) == 0:
        print(f"❌ No numeric feature columns found in {csv_path}, skipping.")
        return None

    # Flatten labels (all blocks)
    y = merged[y_cols].values.reshape(-1)
    X = merged[X_cols].values
    X = np.repeat(X, len(y_cols), axis=0)  # repeat each feature row for all blocks

    # Remove NaNs
    mask = ~np.isnan(y)
    X, y = X[mask], y[mask].astype(int)

    # --- Split train/test ---
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    # --- Scale features ---
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # --- Train XGBoost ---
    model = XGBClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    # --- Evaluate ---
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    metrics = {
        "Feature_Set": feature_name,
        "Train_Accuracy": accuracy_score(y_train, y_pred_train),
        "Train_Precision": precision_score(y_train, y_pred_train, average='macro', zero_division=0),
        "Train_Recall": recall_score(y_train, y_pred_train, average='macro', zero_division=0),
        "Train_F1": f1_score(y_train, y_pred_train, average='macro', zero_division=0),
        "Test_Accuracy": accuracy_score(y_test, y_pred_test),
        "Test_Precision": precision_score(y_test, y_pred_test, average='macro', zero_division=0),
        "Test_Recall": recall_score(y_test, y_pred_test, average='macro', zero_division=0),
        "Test_F1": f1_score(y_test, y_pred_test, average='macro', zero_division=0)
    }

    print(f"✅ {feature_name} metrics: Train F1={metrics['Train_F1']:.3f}, Test F1={metrics['Test_F1']:.3f}")
    return metrics

# --- Evaluate all feature CSVs ---
results = []
for csv_path in sorted(glob.glob(os.path.join(FEATURE_DIR, "*_features.csv"))):
    res = evaluate_feature_set(csv_path, label_df)
    if res is not None:
        results.append(res)

# --- Summary Table ---
results_df = pd.DataFrame(results)
print("\n📊 Feature Comparison Summary:")
print(results_df.sort_values("Test_F1", ascending=False).reset_index(drop=True))

# --- Save results ---
results_df.to_csv("xgboost_feature_comparison.csv", index=False)
print("\n✅ Saved summary to 'xgboost_feature_comparison.csv'")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# --- LOAD FEATURES ---
df = pd.read_csv("features/all_features.csv")

# --- LOAD LABEL FILE ---
labels = pd.read_excel("labels_final.xlsx")  # replace with your actual file
labels["filename"] = labels["filename"].astype(str)

# --- RESHAPE LABEL FILE ---
# Convert block_0 ... block_63 columns into long format
label_long = labels.melt(
    id_vars=["filename"],
    var_name="grid_index",
    value_name="block_label"
)
label_long["grid_index"] = label_long["grid_index"].str.extract("(\d+)").astype(int)

# --- MERGE WITH FEATURES ---
merged = df.merge(label_long, on=["filename", "grid_index"], how="inner")

# --- DEFINE FEATURES + TARGET ---
feature_cols = [c for c in merged.columns if c.startswith("feat_")]
X = merged[feature_cols].values
y = merged["block_label"].values.astype(int)

# --- CLEAN & IMPUTE ---
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
imputer = SimpleImputer(strategy="mean")
X = imputer.fit_transform(X)

# --- SCALE ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- TRAIN-TEST SPLIT ---
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# --- BALANCE USING SMOTE ---
print("Before SMOTE:", np.bincount(y_train))
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
print("After SMOTE:", np.bincount(y_train_bal))

# --- TRAIN XGBOOST ---
model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)
model.fit(X_train_bal, y_train_bal)

# --- EVALUATE ---
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\n🔹 XGBoost + SMOTE Results:")
print(f"Accuracy: {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall: {rec:.3f}")
print(f"F1-score: {f1:.3f}")
print("\nDetailed Report:\n", classification_report(y_test, y_pred, digits=3))

# --- SAVE RESULTS ---
summary = pd.DataFrame([{
    "Method": "XGBoost + SMOTE",
    "Accuracy": acc,
    "Precision": prec,
    "Recall": rec,
    "F1": f1
}])
summary.to_csv("xgboost_smote_results.csv", index=False)

print("\n✅ SMOTE-based evaluation complete → 'xgboost_smote_results.csv'")


In [ ]:
# --- Predictions ---
y_pred_test = xgb.predict(X_test)
y_pred_train = xgb.predict(X_train_bal)

# Use probability of positive class only
y_prob_test = xgb.predict_proba(X_test)[:, 1]
y_prob_train = xgb.predict_proba(X_train_bal)[:, 1]

# --- Metrics function ---
def report(split, y_true, y_pred):
    print(f"\n📌 {split} Metrics:")
    print(classification_report(y_true, y_pred, digits=4))
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision (macro):", precision_score(y_true, y_pred, average='macro'))
    print("Recall (macro):", recall_score(y_true, y_pred, average='macro'))
    print("F1 Score (macro):", f1_score(y_true, y_pred, average='macro'))

report("TRAIN", y_train_bal, y_pred_train)
report("TEST", y_test, y_pred_test)

# --- ROC-AUC ---
auc_train = roc_auc_score(y_train_bal, y_prob_train)
auc_test = roc_auc_score(y_test, y_prob_test)
print(f"\n🔥 Train ROC-AUC: {auc_train:.4f}")
print(f"🔥 Test ROC-AUC: {auc_test:.4f}")

# --- Plot ROC curves ---
plt.figure(figsize=(6,6))
RocCurveDisplay.from_predictions(y_test, y_prob_test, name="XGBoost (Test)", plot_chance_level=True)
plt.title("ROC Curve - Test")
plt.show()

plt.figure(figsize=(6,6))
RocCurveDisplay.from_predictions(y_train_bal, y_prob_train, name="XGBoost (Train)", plot_chance_level=True)
plt.title("ROC Curve - Train")
plt.show()

In [ ]:
import cv2
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

# --- PARAMETERS ---
GRID_ROWS, GRID_COLS = 8, 8
TARGET_W, TARGET_H = 800, 600
HIGHLIGHT_DIR = "xgb_highlighted"

os.makedirs(HIGHLIGHT_DIR, exist_ok=True)

# --- LOAD FEATURES + PREDICTIONS ---
features_df = pd.read_csv("features/all_features.csv")   # features generated earlier
results_df = pd.read_csv("xgboost_smote_predictions.csv")  # should contain filename, grid_index, y_pred

# Ensure correct column names
if "filename" not in results_df.columns or "grid_index" not in results_df.columns:
    raise ValueError("❌ 'xgboost_smote_predictions.csv' must contain 'filename', 'grid_index', and 'y_pred' columns!")

# --- MERGE FEATURES + PREDICTIONS ---
merged = features_df.merge(results_df, on=["filename", "grid_index"], how="left")

def enforce_4_3_aspect_and_scale(img):
    """Ensure 4:3 aspect ratio and resize."""
    h, w = img.shape[:2]
    desired_ratio = 4 / 3
    cur_ratio = w / h
    if abs(cur_ratio - desired_ratio) > 1e-3:
        if cur_ratio > desired_ratio:
            new_w = int(h * desired_ratio)
            x0 = (w - new_w) // 2
            img = img[:, x0:x0 + new_w]
        else:
            new_h = int(w / desired_ratio)
            y0 = (h - new_h) // 2
            img = img[y0:y0 + new_h, :]
    return cv2.resize(img, (TARGET_W, TARGET_H))

def highlight_grid(img, preds):
    """Overlay red (animal) or green (background) highlights."""
    img_vis = img.copy()
    h, w = img.shape[:2]
    cell_h, cell_w = h // GRID_ROWS, w // GRID_COLS
    overlay = img_vis.copy()

    for i in range(GRID_ROWS):
        for j in range(GRID_COLS):
            idx = i * GRID_COLS + j
            if idx >= len(preds):
                continue
            color = (0, 255, 0) if preds[idx] == 0 else (0, 0, 255)
            y0, y1 = i * cell_h, (i + 1) * cell_h
            x0, x1 = j * cell_w, (j + 1) * cell_w
            cv2.rectangle(overlay, (x0, y0), (x1, y1), color, -1)

    cv2.addWeighted(overlay, 0.3, img_vis, 0.7, 0, img_vis)
    return img_vis

# --- LOOP THROUGH IMAGES ---
img_dir = "images"

for fname in merged["filename"].unique():
    img_path = os.path.join(img_dir, fname)
    if not os.path.exists(img_path):
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue

    img = enforce_4_3_aspect_and_scale(img)

    preds = merged.loc[merged["filename"] == fname, "y_pred"].values
    if len(preds) < GRID_ROWS * GRID_COLS:
        continue

    img_high = highlight_grid(img, preds)
    out_path = os.path.join(HIGHLIGHT_DIR, fname)
    cv2.imwrite(out_path, img_high)

    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(img_high, cv2.COLOR_BGR2RGB))
    plt.title(f"Predicted Animal Regions: {fname}")
    plt.axis("off")
    plt.show()

print(f"\n✅ Highlighted images saved in: '{HIGHLIGHT_DIR}/'")
